# Retail EDA and frozen error analysis
Executed 30 October 2026. Re-run from the project or notebooks directory after generating the retail pipeline outputs. No test-window tuning.


In [1]:
from pathlib import Path
import pandas as pd
ROOT=Path.cwd() if (Path.cwd()/'src').exists() else Path.cwd().parent
for name in ['demand-profile.csv','test-month-errors.csv','largest-errors.csv']:
    print(pd.read_csv(ROOT/'reports/retail'/name).to_string(index=False))

product_id  days       mean        std  maximum  total  zero_sales_fraction
     20725   373  40.075067  47.043445      284  14948             0.209115
     22423   373  29.560322  39.592205      324  11026             0.195710
     47566   373  45.453083  64.314717      337  16954             0.222520
    85099B   373 118.353887 174.247131     1533  44146             0.198391
    85123A   373  95.139410 222.674293     3114  35487             0.184987
  month  rows       MAE       RMSE  WAPE_percent       bias
2011-09 150.0 37.066277  59.568326     60.335773  -1.158468
2011-10 155.0 46.471369 129.739500     83.301285  -3.591185
2011-11 150.0 56.728647 150.279299     65.130479 -24.157871
2011-12  40.0 40.986067  73.821072     66.861447   6.134623
      date product_id  actual_units  random_forest_leaf_10        error  absolute_error
2011-10-06     85099B          1533             162.487570 -1370.512430     1370.512430
2011-11-03     85123A          1265              98.697543 -1166.302

# Retail analytical findings - 30 October 2026

Computed from the frozen 26 October model outputs; no test-based model changes.

## Data quality and interpretation
541,909 raw rows include 5,268 exact duplicates, 9,288 cancellations, 10,624 nonpositive-quantity rows and 2,517 nonpositive-price rows. Counts overlap and must not be added as mutually exclusive exclusions. Retain positive UK merchandise sales, exclude cancellations and final partial date, drop exact duplicates as an explicit assumption. Some identical rows may be legitimate repeat line entries. Customer ID has 135,080 missing values and is not used or retained. Missing descriptions are excluded by the preparation filter. 475,216 eligible gross-sales rows precede the five-product selection.

Only recorded gross sales are modeled, not net sales after returns or unconstrained demand. The five products are selected using training invoices only. There are 377 panel rows with no recorded sale; they become zero recorded units, but closures and stockouts cannot be distinguished. The sample cannot justify full-catalog or live-business claims.

## Demand patterns
Training-only weekday profiles avoid using test behavior for feature selection. Weekday differences are descriptive associations; trading hours and their causes are not established by this dataset. Zero days and occasional large orders create intermittent/bursty sales. Full-calendar profiles and plots are descriptive; no new model was fitted from them. One year of data is insufficient to establish repeatable annual seasonality.

## Evaluation and failure cases
Fixed chronological splits and past-only rolling features support a rolling one-day-ahead evaluation. Validation MAE selects Random Forest leaf size 10 before test scoring; train plus validation is then used for the evaluation refit. Test data remain reserved from model selection. The deployment demonstration is a separate all-history refit.

Selected test MAE 46.2864 units, RMSE 116.7474 units, WAPE 68.6412%, bias -8.3004 units. The strongest test baseline is seven-day mean with MAE 59.7804; relative MAE reduction is 22.5727%. This is not classification accuracy. RMSE far above MAE indicates severe large misses. The largest-error table and scatter expose underpredicted order bursts; monthly metrics quantify temporal variation without tuning to it. Unknown promotions, bulk purchases and stock availability prevent causal explanations. Product WAPE may be unstable for low-volume subsets. No confidence intervals or significance claim is made for serially dependent observations.

## Inventory policy and recommendations
Safety stock = z * daily standard deviation * sqrt(lead days). Reorder point = mean daily sales * lead days + safety stock. Position = on hand + on order - backorders. A continuous-review (s,S) trigger at or below the reorder point orders toward mean*(lead+extra coverage)+z*std*sqrt(lead+extra coverage), respecting pack size and minimum order.

All stock, lead time and service factors are user assumptions. This formula assumes stationary independent daily demand, fixed lead time and a normal approximation, which bursty sales may violate. Extra coverage is a cycle target, not a periodic review interval. A one-day ML prediction is never multiplied across supplier lead time. Demand standard deviation is a simple scenario proxy, not a calibrated forecast-error distribution. No measured stockout reduction, service-level guarantee, savings or cost optimum is established.

Before operational use, obtain stock snapshots, receipts, supplier lead-time distributions, prices/promotions and lost-sales indicators. Backtest feasible replenishment policies with holding/stockout/order costs and rolling-origin uncertainty calibration on a new untouched period. Extend SKU coverage and monitor drift, bias and exceptions before any automated purchasing. Current app/API are local research tools without production authentication or multi-user persistence.

## Sources and contribution
Chen, D. (2015), Online Retail, UCI Machine Learning Repository, DOI 10.24432/C5BW33, CC BY 4.0: https://archive.ics.uci.edu/dataset/352/online+retail . Raw workbook SHA-256: 43465a06f2ccf7c8b5bd2892bc7defb52f97487934fe93b16ae4c3936424676d.
Qveto Project 1 brief and SOP are local reference sources; originals are preserved outside the public project repository. Pranav completed the implementation with AI assistance. No unreceived teammate work or independent review is claimed.
